In [ ]:
import tensorflow as tf
import cv2
import os
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
image_array = cv2.imread(r"C:\Users\AAKAR\OneDrive\Documents\REAL_TIME_FACE_MASK_DETECTION\DATASET\with_mask\with_mask_1.jpg")

In [ ]:
plt.imshow(image_array)

In [ ]:
image_array.shape

In [ ]:
data_directory = r"C:\Users\AAKAR\OneDrive\Documents\REAL_TIME_FACE_MASK_DETECTION\DATASET" ## training dataset
Classes = ["with_mask", "without_mask"]  ## list of classes
for category in Classes:
    path = os.path.join(Datadirectory, category)  ## //
    for img in os.listdir(path):
        img_array = cv2.imread(os.path.join(path, img))
        #backtorgb = cv2.cvtColor(img_array, cv2.COLOR_GRAY2RGB)
        plt.imshow(cv2.cvtColor(img_array, cv2.COLOR_BGR2RGB))
        plt.show()
        break
    break

In [ ]:
img_size = 224  ## ImageNet => 224 x 224

new_array = cv2.resize(img_array, (img_size, img_size))
plt.imshow(cv2.cvtColor(new_array, cv2.COLOR_BGR2RGB))
plt.show()

In [ ]:
training_Data = []  ## data

def create_training_Data():
    for category in Classes:
        path = os.path.join(data_directory, category)
        class_num = Classes.index(category)  ## 0 1,  ## Label
        for img in os.listdir(path):
            try:
                img_array = cv2.imread(os.path.join(path, img))
                new_array = cv2.resize(img_array, (img_size, img_size))
                training_Data.append([new_array, class_num])
            except Exception as e:
                pass

create_training_Data()

In [ ]:
print(len(training_Data))

In [ ]:
import random
random.shuffle(training_Data)

In [ ]:
X = []  ## data/feature
y = []  ## label

for features, label in training_Data:
    X.append(features)
    y.append(label)

X = np.array(X).reshape(-1, img_size, img_size, 3)

In [ ]:
X.shape

In [ ]:
y = np.array(y)
X = X / 255.0

In [ ]:
y[1000]

In [ ]:
Y = np.array(y)

import pickle

pickle_out = open(r"C:\Users\AAKAR\OneDrive\Documents\REAL_TIME_FACE_MASK_DETECTION\FEATURES_TARGET\X.pickle", "wb")
pickle.dump(X, pickle_out)
pickle_out.close()

pickle_out = open(r"C:\Users\AAKAR\OneDrive\Documents\REAL_TIME_FACE_MASK_DETECTION\FEATURES_TARGET\y.pickle", "wb")
pickle.dump(y, pickle_out)
pickle_out.close()

In [ ]:
pickle_in = open(r"C:\Users\AAKAR\OneDrive\Documents\REAL_TIME_FACE_MASK_DETECTION\FEATURES_TARGET\X.pickle", "rb")
X = pickle.load(pickle_in)

pickle_in = open(r"C:\Users\AAKAR\OneDrive\Documents\REAL_TIME_FACE_MASK_DETECTION\FEATURES_TARGET\y.pickle", "rb")
y = pickle.load(pickle_in)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
model = tf.keras.applications.mobilenet.MobileNet()

In [ ]:
model.summary()

In [ ]:
base_input = model.input   ## instead of model.layers[0].input
base_output = model.layers[-4].output

Flat_layer = layers.Flatten()(base_output)
final_output = layers.Dense(1)(Flat_layer)  ## 0, 1
final_output = layers.Activation('sigmoid')(final_output)  ## typo bhi fix kar diya

new_model = keras.Model(inputs=base_input, outputs=final_output)

In [ ]:
new_model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
new_model.fit(X, Y, epochs=10, validation_split=0.1)

175/214 ━━━━━━━━━━━━━━━━━━━━ 1:32 2s/step - accuracy: 0.9716 - loss: 0.0921

In [ ]:
new_model.save(r"C:\Users\AAKAR\OneDrive\Documents\REAL_TIME_FACE_MASK_DETECTION\facemask_model.h5')

In [ ]:
new_model = tf.keras.models.load_model(r"C:\Users\AAKAR\OneDrive\Documents\REAL_TIME_FACE_MASK_DETECTION\facemask_model.h5')

In [ ]:
frame = cv2.imread(r"C:\Users\AAKAR\OneDrive\Documents\REAL_TIME_FACE_MASK_DETECTION\DATASET\with_mask\with_mask_2.jpg")

In [ ]:
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

In [ ]:
final_image = cv2.resize(frame, (224, 224))
final_image = np.expand_dims(final_image, axis=0)  ## need fourth dimension
final_image = final_image / 255.0

In [ ]:
predictions = new_model.predict(final_image)
print(predictions)

if predictions[0][0] > 0.5:
    print("No Mask")  
else:
    print("Face Mask")